# Implement an analysis step, step by step

You write a small step, run it through the engine, give it its own conda environment, and add a second step that waits for a group of images. The [README](README.md) of this part is the reference.

A step is one Python file with one function, `run`. The engine hands it a dictionary with the image. The step adds its result under its own name and returns the dictionary. A recipe lists which steps run in which order. That is the whole contract.

Everything here runs on a laptop. No GPU.

## Before you start

- ZMART Analysis installed as in [Install it](../../README.md#install-it).
- `pip install numpy jupyter`.

Run the cells one at a time. The first one moves to the root of the repository and makes the workflow folder, with the four folders every workflow has.

In [1]:
import os
while not os.path.exists("pyproject.toml"):
    os.chdir("..")

for folder in ("pipelines", "steps", "environments", "tests"):
    os.makedirs(f"workflows/intensity/{folder}", exist_ok=True)
print(os.getcwd())

/home/user/ZMART-analysis


## 1. Write the step

The image comes in under `pipeline_data["input"]`: exactly what will be passed to `engine.submit`. The result goes under the step's own name. The step returns the whole dictionary, so the next step sees everything.

In [2]:
%%writefile workflows/intensity/steps/mean_intensity.py
"""mean_intensity -- the mean and the brightest pixel of one image."""

import numpy as np


def run(pipeline_data, state, **params):
    image = np.asarray(pipeline_data["input"]["image"], dtype=float)
    pipeline_data["mean_intensity"] = {
        "mean": float(image.mean()),
        "max": float(image.max()),
    }
    return pipeline_data

Writing workflows/intensity/steps/mean_intensity.py


## 2. Call it directly

A step is a plain function. Call it before the engine does. The `{}` is `state`, a dictionary the step may keep things in between jobs.

In [3]:
import sys
import numpy as np

sys.path.insert(0, "workflows/intensity/steps")
from mean_intensity import run

out = run({"input": {"image": np.arange(16).reshape(4, 4)}}, {})
print(out["mean_intensity"])

{'mean': 7.5, 'max': 15.0}


A test calls the step the same way, with an `assert`, in `workflows/intensity/tests/test_mean_intensity.py`.

## 3. Write the recipe

The first key after `metadata` names the workflow. Under it, the steps in order. `mean_intensity:` with nothing under it means no parameters.

In [4]:
%%writefile workflows/intensity/pipelines/intensity.yaml
# The mean and the brightest pixel of every image.

metadata:
  purpose: "Mean intensity per image"
  version: "1.0"
  functions_dir: "../steps"

intensity:
  - mean_intensity:

Writing workflows/intensity/pipelines/intensity.yaml


## 4. Run it through the engine

There is no `METADATA` yet, so the step runs in the environment you are in. `shutdown(wait=True)` waits for every job.

In [5]:
from engine import Engine

engine = Engine()
engine.register("intensity", "workflows/intensity/pipelines/intensity.yaml")

rng = np.random.default_rng(0)
for i in range(3):
    engine.submit("intensity", {"image": rng.integers(0, 4096, (64, 64)), "name": f"image_{i}"})

engine.shutdown(wait=True)
for result in engine.results("intensity"):
    print(result["input"]["name"], result["mean_intensity"])

image_0 {'mean': 2062.80224609375, 'max': 4095.0}
image_1 {'mean': 2022.114501953125, 'max': 4095.0}
image_2 {'mean': 2060.589599609375, 'max': 4095.0}


Look at one result. The engine added where the step ran and which packages it imported. Your environment name and versions will differ. That is the point.

In [6]:
print(result["provenance"]["mean_intensity"])

{'environment': 'ZMART--intensity--main', 'python': '3.13.16', 'fingerprint': '349cc8f2d2ee97e8', 'packages': {'numpy': '2.5.3', 'setuptools': '68.1.2'}}


## 5. Add a parameter

Make the brightest pixel a percentile, so one hot pixel cannot decide it:

In [7]:
%%writefile workflows/intensity/steps/mean_intensity.py
"""mean_intensity -- the mean and a brightness percentile of one image."""

import numpy as np


def run(pipeline_data, state, **params):
    percentile = float(params.get("percentile", 100))
    image = np.asarray(pipeline_data["input"]["image"], dtype=float)
    pipeline_data["mean_intensity"] = {
        "mean": float(image.mean()),
        "max": float(np.percentile(image, percentile)),
        "percentile": percentile,
    }
    return pipeline_data

Overwriting workflows/intensity/steps/mean_intensity.py


In the recipe, with its default and a comment. Parameters under a step arrive as keyword arguments. The default in the recipe makes the recipe a complete record of what was done.

In [8]:
%%writefile workflows/intensity/pipelines/intensity.yaml
# The mean and a brightness percentile of every image.

metadata:
  purpose: "Mean intensity per image"
  version: "1.0"
  functions_dir: "../steps"

intensity:
  - mean_intensity:
      percentile: 99.9    # the brightest this share of pixels; 100 is the true maximum

Overwriting workflows/intensity/pipelines/intensity.yaml


## 6. Give it its own environment

Name it in the step file, above `run`. The name is `ZMART--<workflow>--<step>`. With one environment, call it `main`. `max_workers: 4` lets four copies run at once. The engine reads this as text, so it must be a plain literal.

In [9]:
%%writefile workflows/intensity/steps/mean_intensity.py
"""mean_intensity -- the mean and a brightness percentile of one image."""

import numpy as np

METADATA = {
    "description": "Mean intensity and a brightness percentile of one image",
    "version": "1.0",
    "environment": "ZMART--intensity--main",
    "max_workers": 4,
}


def run(pipeline_data, state, **params):
    percentile = float(params.get("percentile", 100))
    image = np.asarray(pipeline_data["input"]["image"], dtype=float)
    pipeline_data["mean_intensity"] = {
        "mean": float(image.mean()),
        "max": float(np.percentile(image, percentile)),
        "percentile": percentile,
    }
    return pipeline_data

Overwriting workflows/intensity/steps/mean_intensity.py


Then the two environment scripts. `setup_env.py` lists the packages; `clean_env.py` removes the environment again.

In [10]:
%%writefile workflows/intensity/environments/setup_env.py
"""Create the conda environment for the intensity workflow."""

import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[3] / "engine"))
from conda_utils import setup_workflow_env  # noqa: E402

if __name__ == "__main__":
    setup_workflow_env(
        workflow="intensity",
        pip_packages=["numpy"],
        diagnostics=[("numpy imports", "import numpy; print(numpy.__version__)")],
        install_torch=False,
    )

Writing workflows/intensity/environments/setup_env.py


In [11]:
%%writefile workflows/intensity/environments/clean_env.py
"""Remove the conda environments of the intensity workflow."""

import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[3] / "engine"))
from conda_utils import clean_workflow_envs  # noqa: E402

if __name__ == "__main__":
    clean_workflow_envs(workflow="intensity")

Writing workflows/intensity/environments/clean_env.py


Run the setup once. It takes a minute.

In [ ]:
!python workflows/intensity/environments/setup_env.py

Now run the engine again. The Python is the same as in step 4. The output is not: `max` is now the 99.9th percentile, and the results come back in another order, because `max_workers: 4` ran the three images at once. `input["name"]` says which is which.

In [12]:
from engine import Engine

engine = Engine()
engine.register("intensity", "workflows/intensity/pipelines/intensity.yaml")

rng = np.random.default_rng(0)
for i in range(3):
    engine.submit("intensity", {"image": rng.integers(0, 4096, (64, 64)), "name": f"image_{i}"})

engine.shutdown(wait=True)
for result in engine.results("intensity"):
    print(result["input"]["name"], result["mean_intensity"])

image_0 {'mean': 2062.80224609375, 'max': 4092.9050000000007, 'percentile': 99.9}
image_2 {'mean': 2060.589599609375, 'max': 4092.0, 'percentile': 99.9}
image_1 {'mean': 2022.114501953125, 'max': 4093.8100000000013, 'percentile': 99.9}


And the provenance names the new environment:

In [13]:
print(result["provenance"]["mean_intensity"])

{'environment': 'ZMART--intensity--main', 'python': '3.13.16', 'fingerprint': '349cc8f2d2ee97e8', 'packages': {'numpy': '2.5.3', 'setuptools': '68.1.2'}}


The worker stays running between images. A model loaded on the first image and kept in `state` is still there for the second.

## 7. A step over a group

How does the intensity vary across a group of images? That can only be answered once the whole group is in.

Two differences from a per-image step. It receives `pipeline_data["results"]`, one entry per image in the group. And it returns a *new* dictionary, not `pipeline_data`.

In [14]:
%%writefile workflows/intensity/steps/summarise_group.py
"""summarise_group -- the spread of mean intensity over one completed group."""

import numpy as np

METADATA = {
    "description": "Mean and spread of intensity over a group of images",
    "version": "1.0",
    "environment": "ZMART--intensity--main",
}


def run(pipeline_data, state, **params):
    means = [r["mean_intensity"]["mean"] for r in pipeline_data["results"]]
    return {
        "group": pipeline_data["metadata"]["unit"],
        "n_images": len(means),
        "n_failed": len(pipeline_data["failures"]),
        "mean": float(np.mean(means)) if means else None,
        "spread": float(np.std(means)) if means else None,
    }

Writing workflows/intensity/steps/summarise_group.py


Add it to the recipe with a `scope`:

In [15]:
%%writefile workflows/intensity/pipelines/intensity.yaml
# The mean and a brightness percentile of every image, and their spread per group.

metadata:
  purpose: "Mean intensity per image, spread per group"
  version: "1.0"
  functions_dir: "../steps"

intensity:
  - mean_intensity:
      percentile: 99.9    # the brightest this share of pixels; 100 is the true maximum
  - summarise_group:
      scope: group

Overwriting workflows/intensity/pipelines/intensity.yaml


Submit images with a `scope`, and say when a group is complete:

In [16]:
engine = Engine()
engine.register("intensity", "workflows/intensity/pipelines/intensity.yaml")

for group in (1, 2):
    for i in range(4):
        last = i == 3
        engine.submit(
            "intensity",
            {"image": rng.integers(0, 4096, (64, 64)), "name": f"g{group}_image_{i}"},
            scope={"group": group},
            complete="group" if last else None,
        )

engine.shutdown(wait=True)
for result in engine.results("intensity"):
    if result.get("_scope_level") == "group":
        print(result["group"], result["n_images"], round(result["spread"], 1))

{'group': 2} 4 13.4
{'group': 1} 4 15.8


`mean_intensity` ran on every image as it was submitted. `summarise_group` ran once per group, after each `complete="group"`. The engine never guesses when a group is done. The acquisition says so.

## 8. Clean up

Remove the environment, then the folder. Or keep the folder as the start of a real workflow.

In [ ]:
!python workflows/intensity/environments/clean_env.py

In [17]:
import shutil
shutil.rmtree("workflows/intensity")

## When something goes wrong

- **`Step 'x' sets 'environment' in the pipeline YAML`** at `register`. The environment belongs in the step's `METADATA`, not in the recipe.
- **A job fails; the others carry on.** `engine.status("intensity")` lists the failures with the step, the message and the traceback.
- **`WorkerSpawnError` naming an environment.** It does not exist, or conda cannot be found. Run `setup_env.py`, or `setup_env.py --check`.
- **The step runs old code.** A worker keeps the module it imported. After editing a step, `engine.shutdown()` and make a new `Engine`.
- **The scoped step never runs.** No `complete="group"` was sent, or it was sent with a different `scope` than the images.

## Next

- The [README](README.md) of this part: every key of `pipeline_data` and `METADATA`, overrides per submission, provenance, tests.
- [Use the engine](../1_use_the_engine/README.md): every `Engine` call.
- `workflows/focus/steps/score_focus.py`: a real step with its own environment and a test.
- `workflows/object_analysis/steps/summarise_population.py`: a real step over a scope.

---

MIT license. Thom de Hoog, Center for Microscopy and Image Analysis (ZMB), University of Zurich. thom.dehoog@zmb.uzh.ch, thomdehoog@gmail.com.